In [8]:
# install the necessary dependencies
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [9]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [10]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [11]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])

    return json.loads(text)



In [12]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [13]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output


In [14]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [15]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [16]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [17]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket ARN Region Extractor\n\nHere's a solution with multiple approaches:\n\n```python\nimport re\n\ndef extract_region_from_s3_arn_v1(arn: str) -> str | None:\n    \"\"\"\n    Extract AWS region from S3 bucket ARN using regex.\n    \n    S3 ARN format: arn:aws:s3:::bucket-name\n    Region is typically in the bucket name itself (e.g., my-bucket-us-east-1)\n    \n    Args:\n        arn: The S3 bucket ARN string\n        \n    Returns:\n        The region string or None if not found\n    \"\"\"\n    if not arn or not isinstance(arn, str):\n        return None\n    \n    # Match AWS region patterns at the end of the bucket name\n    # Valid region formats: us-east-1, eu-west-2, ap-southeast-1, etc.\n    region_pattern = r'(us|eu|ap|ca|sa|me|af)-[a-z]+-\\d+$'\n    \n    # Extract bucket name from ARN (format: arn:aws:s3:::bucket-name)\n    match = re.search(r'arn:aws:s3:::([^/:]+)', arn)\n    if not match:\n        return None\n    \n    bucket_name = match.g